# Incremental web components in solveit

We want to author an interactive tutorial *one piece at a time*: render a custom tag in one cell, then later add its styles, then later add its behaviour - all against the live page, with no reload.

Plain custom elements fight this. `customElements.define` can only run *once* per tag name, and once defined the class is frozen, so "add more styles/behaviour later" is impossible.

The fix is a **registry-driven** element. We define `my-thing` *once* with a generic `render()` that reads from a global JS table. Adding styles or callbacks just pushes into that table and re-renders the live instances - no redefine errors, survives cell re-runs, genuinely incremental.

First the imports we need.

In [ ]:
from fasthtml.common import *
from fasthtml.jupyter import *

The one-time runtime: `shadowReg` defines a tag whose `render()` pulls styles and callbacks from a global `SH` table, so we can keep adding to it later.

In [ ]:
shadow_runtime = Script("""
window.SH=window.SH||{};
function shadowReg(nm){
  if(customElements.get(nm)) return;
  class C extends HTMLElement{
    connectedCallback(){ this.attachShadow({mode:'open'}); this.render(); }
    render(){ let r=SH[nm]||{styles:[],cbs:[]}; this.shadowRoot.innerHTML=`<style>${r.styles.join('\\n')}</style>`+this.innerHTML; r.cbs.forEach(f=>f(this)); }
  }
  customElements.define(nm,C);
}
""")

`add_shadow_styles` registers the tag (once) then pushes a CSS string into its table and re-renders any live instances.

In [ ]:
def _push(nm, key, val): return Script(f"SH['{nm}']=SH['{nm}']||{{styles:[],cbs:[]}};SH['{nm}'].{key}.push({val});document.querySelectorAll('{nm}').forEach(e=>e.render&&e.render());")

@patch
def add_shadow_styles(self:FT, css): return Div(Script(f"shadowReg('{self.tag}')"), _push(self.tag, 'styles', f"`{css}`"), self)

Now the demo: render `my-thing` and give it some encapsulated styles. Re-running this cell is safe, and a later cell can push more styles to the same tag.

In [ ]:
show(shadow_runtime, Ft_hx('my-thing', "hello shadow").add_shadow_styles(":host{display:block;padding:1rem;background:#F0EBE0;border-radius:8px;color:#86744F;}"))